In [ ]:
import os
from src.neo4j_manager import Neo4jManager
from src.mineru_parser import MinerUParser
from src.utils.schema_loader import DomainSchema
from src.hybrid_extractor import HybridEntityExtractor
from src.multimodal_grounding import MultimodalGrounder
from src.hybrid_retrieval import HybridRetriever
import src.utils.constants as const

from langchain_ollama import OllamaLLM

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = const.TEST_PDF
neo4j_config = {
	"uri": const.NEO4J_URI,
	"username": const.NEO4J_USERNAME,
	"password": const.NEO4J_PASSWORD
}
schema_path = const.CONFIG_DIR + const.SCHEMA_FILE

llm = OllamaLLM(model="llama3.1:8b", temperature=0)

query = "What malware is mentioned in the document?"

In [3]:
# Step 1: Parse PDF document (using MinerU parser)
parser = MinerUParser({})
parsed_doc = parser.parse(pdf_path)

Predict: 100%|██████████| 7/7 [00:34<00:00,  4.94s/it]
2025-11-03 18:00:37,870 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page2_0_NCHP 50 Screenshot GinWui Rootkit.png
2025-11-03 18:00:37,887 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page3_1_image_1.png
2025-11-03 18:00:37,917 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page5_2_image_2.png
2025-11-03 18:00:37,935 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page6_3_Wicked Roses Website wwwmghackercom.png
2025-11-03 18:00:37,949 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page6_4_NCPH Studio website wwwncphnet.png
2025-11-03 18:00:38,001 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page7_5_Zigong Sichuan Province in south-central China.png
2025-11-03 18:00:38,040 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_and

In [4]:
# Step 2: Load domain schema
domain_schema = DomainSchema(schema_file=schema_path)

In [5]:
# Step 3: Initialize Neo4jManager
neo4j_manager = Neo4jManager(
    uri=neo4j_config['uri'],
    username=neo4j_config['username'],
    password=neo4j_config['password'],
    database=neo4j_config.get('database', 'neo4j')
)

2025-11-03 18:00:38,170 - Neo4jManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
2025-11-03 18:00:38,179 - Neo4jManager - INFO - Neo4j schema initialized
2025-11-03 18:00:38,184 - Neo4jManager - ERROR - Error creating vector index: {code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `gds.index.createVectorSimilarityIndex` registered for this database instance. Please ensure you've spelled the procedure name correctly and that the procedure is properly deployed.}


In [6]:
# Step 4: Extract entities and relations via Hybrid Extractor (pattern + LLM)
extractor = HybridEntityExtractor(
    config={"spacy_model": "en_core_web_lg",
            "use_llm_entities": True, "use_llm_relations": True},
    domain_schema=domain_schema,
    llm=llm
)

In [7]:
entities = extractor.extract_entities(parsed_doc.text)
relations = extractor.extract_relations(parsed_doc.text, entities)

c:\Users\ishani.kathuria\projects\TrustworthyRAG\src\hybrid_extractor.py:78: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  llm_response = self.llm(prompt)
2025-11-03 18:00:47,510 - HybridEntityExtractor - INFO - LLM response:
[
  {
    "text": "Wicked Rose",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 10
  },
  {
    "text": "KuNgBiM",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 9
  },
  {
    "text": "Charles",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 8
  },
  {
    "text": "Ronag",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 6
  },
  {
    "text": "WHG",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 3
  },
  {
    "text": "WZT",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 3
  },
  {
    "text": "Li0n",
    "label": "PERSON",
    "start_pos": 0,
    "end_pos": 4
  },
  {
    "text": "Wicked R

In [16]:
# Step 5: Ingest entities and relations into Neo4j
neo4j_manager.ingest_entities(entities)
neo4j_manager.ingest_relations(relations)

2025-11-03 18:01:46,654 - Neo4jManager - INFO - Ingested 121 entities
INFO:Neo4jManager:Ingested 121 entities
2025-11-03 18:01:46,792 - Neo4jManager - INFO - Ingested 275 relations
INFO:Neo4jManager:Ingested 275 relations


{'created': 275, 'errors': 0}

In [17]:
# Step 6: Multimodal Grounding - embed images and link to entities
multimodal_grounder = MultimodalGrounder(
    neo4j_manager=neo4j_manager,
	# similarity_threshold=0.1
)

# Gather image files and captions extracted in parsing step
image_paths = [img['path'] for img in parsed_doc.images]
caption_map = {img['path']: img.get('caption', '')
               for img in parsed_doc.images}

multimodal_grounder.process(entities, image_paths, caption_map, parsed_doc.text)

Encriched texts for embedding: [' ## "Wicked Rose" and the NCPH Hacking Group  by Ken Dunham & Jim Melnick Zero-day attacks, where an attack occurs before public knowledge of a ', ' ## "Wicked Rose" and the NCPH Hacking Group  by Ken Dunham & Jim Melnick Zero-day attacks, where an attack occurs before public knowledge of a vulnerability is ', ' ## "Wicked Rose" and the NCPH Hacking Group  by Ken Dunham & Jim Melnick Zero-day attacks, where an attack occurs before public knowledge of a vulnerability is known, is a ', ' ## "Wicked Rose" and the NCPH Hacking Group  by Ken Dunham & Jim Melnick Zero-day attacks, where an attack occurs before public knowledge of a vulnerability is known, is a growing c', 'ionals in the \\(21^{\\text{st}}\\) century. An unprecedented number of zero-day attacks took place in 2006, largely involving Microsoft Office Files. Ken Dunham, Director of the Rapid Response Team, and Jim', 't{st}}\\) century. An unprecedented number of zero-day attacks took place in 200

In [18]:
# Optional: Get stats
stats = neo4j_manager.get_statistics()
print(f"Neo4j Stats: {stats}")

Neo4j Stats: {'total_entities': 121, 'total_relations': 275, 'entity_types': [{'type': 'DATE', 'count': 61}, {'type': 'ORGANIZATION', 'count': 35}, {'type': 'PERSON', 'count': 14}, {'type': 'MALWARE', 'count': 5}, {'type': 'WEBSITE', 'count': 5}, {'type': 'VULNERABILITY', 'count': 1}], 'relation_types': [{'type': 'EXPLOITS', 'count': 172}, {'type': 'CREATED_BY', 'count': 100}, {'type': 'WEBSITE', 'count': 3}]}


In [19]:
# Step 7: Setup Retriever
retrieval_cfg = {
    "embedding_model": "all-MiniLM-L6-v2",
    "milvus": {
        "collection_name": "entity_embeddings",
        "host": "localhost",
        "port": "19530"
    },
    "llm_model": "llama3.1:8b",
    "llm_base_url": "http://localhost:11434"
}
retriever = HybridRetriever(neo4j_manager, retrieval_cfg)

2025-11-03 18:01:57,967 - HybridRetriever - INFO - HybridRetriever components initialized
INFO:HybridRetriever:HybridRetriever components initialized


In [20]:
# Step 8: Retrieve multimodal context (vector+graph)
retrieval_result = retriever.retrieve(query, method="hybrid", top_k=10)
if not retrieval_result.get('success'):
    print(f"Retrieval failed: {retrieval_result.get('error')}")

2025-11-03 18:01:58,697 - HybridRetriever - ERROR - Vector retrieval failed: 'list' object has no attribute 'tolist'
ERROR:HybridRetriever:Vector retrieval failed: 'list' object has no attribute 'tolist'


In [21]:
# Step 9: Format context for prompt (simple concatenation here)
context_str = ""
for item in retrieval_result['results']:
    src = item.get('source', 'unknown')
    txt = item.get('text') or item.get('content') or ''
    score = item.get('score') or item.get('similarity_score') or 'N/A'
    context_str += f"\nSource ({src}) Score [{score}]:\n{txt}\n"

prompt = f"""
You are an expert cybersecurity analyst. Use the following context to answer the question.
Context: {context_str}

Question: {query}

Answer:
"""

In [22]:
# Step 10: Generate answer from LLM
answer = llm.invoke(prompt)

print("\n=== Question ===")
print(query)
print("\n=== Generated Answer ===")
print(answer)
print("\n=== Retrieval Context ===")
print(context_str)


=== Question ===
What malware is mentioned in the document?

=== Generated Answer ===
There is no mention of malware in the provided context. The text appears to be a list of source scores with various phrases or keywords, but none of them specifically refer to malware.

=== Retrieval Context ===

Source (graph) Score [1.0671522617340088]:
the day

Source (graph) Score [0.9087377786636353]:
the next year

Source (graph) Score [0.7912759780883789]:
the last two years

Source (graph) Score [0.7912759780883789]:
the end of 2006

Source (graph) Score [0.7912759780883789]:
the summer of 2006

Source (graph) Score [0.7912759780883789]:
the spring of 2006

Source (graph) Score [0.7912759780883789]:
the Internet Storm Center

Source (graph) Score [0.7912759780883789]:
the Rapid Response Team

Source (graph) Score [0.7912759780883789]:
the Christmas holiday season

Source (graph) Score [0.7912759780883789]:
the NCPH Hacking Group



In [15]:
# Close Neo4j session
# neo4j_manager.close()